In [1]:
# Install required libraries
!pip install folium pandas numpy tensorflow scikit-learn

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import folium
from folium.plugins import HeatMap

print("Libraries imported successfully! TensorFlow version:", tf.__version__)

2026-03-11 12:57:51.324742: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773233871.559980      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773233871.636008      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773233872.290211      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773233872.290258      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773233872.290263      55 computation_placer.cc:177] computation placer alr

Libraries imported successfully! TensorFlow version: 2.19.0


In [2]:
np.random.seed(42)

# Dhaka Regions and Coordinates
regions = ['Mirpur', 'Dhanmondi', 'Gulshan', 'Uttara', 'Mohammadpur', 'Badda', 'Old Dhaka', 'Banani']
lats = [23.8223, 23.7461, 23.7925, 23.8759, 23.7658, 23.7805, 23.7104, 23.7940]
lons = [90.3654, 90.3742, 90.4078, 90.3993, 90.3584, 90.4267, 90.3989, 90.4007]

# Generate 12 months of data (simulating an upward seasonal trend with some random noise)
history = np.zeros((8, 12))
for i in range(8):
    base_cases = np.random.randint(50, 150)
    for month in range(12):
        # Add a growth factor + random fluctuations
        history[i, month] = base_cases + (month * np.random.randint(10, 30)) + np.random.randint(-20, 20)
        history[i, month] = max(0, history[i, month]) # Ensure no negative cases

# Create a DataFrame for easy viewing
columns = [f'Month_{i+1}' for i in range(12)]
df = pd.DataFrame(history, columns=columns)
df.insert(0, 'Region', regions)
df.insert(1, 'Lat', lats)
df.insert(2, 'Lon', lons)

# The "Current" cases are Month 12
df['Current_Cases'] = df['Month_12']

print("12-Month Historical Data Generated:")
display(df[['Region', 'Month_1', 'Month_6', 'Month_11', 'Current_Cases']])

12-Month Historical Data Generated:


,Region,Month_1,Month_6,Month_11,Current_Cases
0,Mirpur,88.0,159.0,367.0,317.0
1,Dhanmondi,103.0,212.0,335.0,271.0
2,Gulshan,38.0,178.0,296.0,303.0
3,Uttara,44.0,136.0,186.0,341.0
4,Mohammadpur,125.0,151.0,293.0,430.0
5,Badda,110.0,182.0,207.0,412.0
6,Old Dhaka,37.0,133.0,169.0,217.0
7,Banani,144.0,229.0,265.0,287.0


In [3]:
# 1. Prepare Data for LSTM (Sequence formulation)
lookback = 3 # The model looks at 3 months to predict the 4th
X_train, y_train = [], []

# Extract sequences from our 12-month history
for r in range(8): # For each region
    for t in range(12 - lookback):
        X_train.append(history[r, t:t+lookback])
        y_train.append(history[r, t+lookback])

# Reshape for LSTM [samples, time_steps, features]
X_train = np.array(X_train).reshape(-1, lookback, 1)
y_train = np.array(y_train)

# 2. Build the LSTM Architecture
model = Sequential([
    LSTM(50, activation='relu', input_shape=(lookback, 1)),
    Dense(25, activation='relu'),
    Dense(1) # Output layer: 1 predicted value
])

model.compile(optimizer='adam', loss='mse')

print("Training LSTM Model... Please wait.")
# Train the model
model.fit(X_train, y_train, epochs=200, verbose=0)
print("Model Training Complete!")

# 3. Predict Future Trends (Month 13)
# To predict Month 13, we give the model the data from Months 10, 11, and 12
X_future = np.array([history[r, -lookback:] for r in range(8)]).reshape(8, lookback, 1)

# Generate predictions and add to dataframe
predictions = model.predict(X_future, verbose=0).flatten()
df['Future_Cases_Predicted'] = [max(0, int(p)) for p in predictions]

print("\nLSTM Predictions for Next Month:")
display(df[['Region', 'Current_Cases', 'Future_Cases_Predicted']])

2026-03-11 12:58:53.469425: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training LSTM Model... Please wait.
Model Training Complete!

LSTM Predictions for Next Month:


,Region,Current_Cases,Future_Cases_Predicted
0,Mirpur,317.0,363
1,Dhanmondi,271.0,425
2,Gulshan,303.0,315
3,Uttara,341.0,295
4,Mohammadpur,430.0,388
5,Badda,412.0,348
6,Old Dhaka,217.0,307
7,Banani,287.0,300


In [4]:
# Create base map centered on Dhaka
map_current = folium.Map(location=[23.7800, 90.4125], zoom_start=12, tiles='CartoDB dark_matter')

# Format data: [Lat, Lon, Weight] (Divided by 20 to normalize heatmap visual scale)
heat_data_current = [[row['Lat'], row['Lon'], row['Current_Cases']/20] for index, row in df.iterrows()]

# Add HeatMap
HeatMap(heat_data_current, radius=40, blur=25, gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}).add_to(map_current)

# Add markers
for index, row in df.iterrows():
    folium.CircleMarker(location=[row['Lat'], row['Lon']], radius=2, color='white', 
                        tooltip=f"{row['Region']} Current: {int(row['Current_Cases'])}").add_to(map_current)

print("CURRENT Heatmap (Month 12 Data):")
display(map_current)

CURRENT Heatmap (Month 12 Data):


In [5]:
# Create new base map
map_future = folium.Map(location=[23.7800, 90.4125], zoom_start=12, tiles='CartoDB dark_matter')

# Format PREDICTED data: [Lat, Lon, Weight]
heat_data_future = [[row['Lat'], row['Lon'], row['Future_Cases_Predicted']/20] for index, row in df.iterrows()]

# Add HeatMap (Using a slightly different gradient to signify it's a prediction)
HeatMap(heat_data_future, radius=40, blur=25, gradient={0.4: 'purple', 0.65: 'orange', 1: 'red'}).add_to(map_future)

# Add markers showing predicted values
for index, row in df.iterrows():
    folium.Marker(
        location=[row['Lat'], row['Lon']],
        tooltip=f"⚠ {row['Region']} Predicted: {row['Future_Cases_Predicted']} cases",
        icon=folium.Icon(color='red', icon='fire')
    ).add_to(map_future)

print("FUTURE TRENDS Heatmap (LSTM Predicted Month 13 Data):")
display(map_future)

FUTURE TRENDS Heatmap (LSTM Predicted Month 13 Data):
